# Card-game action selection

**Recipe 36** · Level 3 (Intermediate) · Decision type: `Choice`

Select a legal action in a turn-based card-game simulator using structured state and fixed action descriptions while Python enforces game rules.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S04: [TypeSafe AI: How to build with TypeSafe](https://docs.typesafe.ai/concepts/how-to-build-with-system-one)
- S07: [TypeSafe AI: Jev 1.13 jaggedness](https://docs.typesafe.ai/model-jaggedness/jev-1.13)

## What you will build

A two-player card game ("Card Duel"): each turn, Python enumerates exactly the cards left in the player's hand and offers them as a `Choice`; Jev picks one to bid for that round's prize, and the simulator plays it out. Python, never the model, enforces which bids are legal, resolves the one-card forced turn at the end of every game without asking a question at all, and decides what happens when the model's answer is not confident enough to trust. By the end you will have seen one typed answer up close, a per-turn evaluation against a rule-based baseline, and a task-level evaluation that plays whole seeded games under three different policies and reports who wins.


## Setup and run mode

The notebook runs from its own folder with the standard library and `jev_cookbook` only. There are no stored responses to replay: `helpers.script` stands in for the model, seeded so the same request always gets the same answer (`get_backend(script=..., seed=...)`). The header below states the mode; every number this notebook prints is a check that the pipeline works, not a measurement of Jev, and says so.


In [ ]:
from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    accuracy,
    confusion_matrix,
    evaluate_selective,
    select_confidence_threshold,
    selective_curve,
)
from jev_cookbook.fixtures import load_inputs, load_labels, select_split
from jev_cookbook.style import (
    apply_style,
    plot_answer_probabilities,
    plot_confusion_matrix,
    plot_risk_coverage,
    run_header,
    show_answer,
)

apply_style()
helpers = load_helpers()  # this recipe's helpers.py, loaded by path
examples = load_inputs()
labels = load_labels()
backend = get_backend(script=helpers.script, seed=helpers.SEED)
# scored excludes the two demo turns: they carry no gold label, so no metric below is about
# them (see "gold label" under Evaluation).
scored = [e for e in examples if e.split not in ("train", "demo")]

offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
# CONTRIBUTING.md section 2: a number from the validation split is a selection step, not a
# result. This label does not depend on the run mode, unlike `check` above.
selection = " (a selection step, not a reported result)"

run_header(
    36,
    "Card-game action selection",
    backend=backend,
    n_examples=len(scored),
)

## The state

Each scored turn in `fixtures/inputs.jsonl` carries `fields`: the game's seed, the turn number, the current prize, the player's remaining hand, and the running score. `build_state` passes on everything except the game seed, which is a Python bookkeeping identifier used only to reproduce the example and never part of the judgment: Jev sees the prize, the hand and the score, never which seeded game produced them.


In [ ]:
demo = [e for e in examples if e.split == "demo"]
if len(demo) < 2:
    raise ValueError("add at least two demo turns to ROWS in build_fixtures.py")
first_turn, second_turn = demo[0], demo[1]
print("Python has:")
print(first_turn.fields)
state = helpers.build_state(first_turn.fields)
print("Jev sees:")
print(state)

## The questions

There is one `Choice` per turn, and its option list is not fixed across the notebook: Python enumerates exactly the cards left in the player's hand this turn and offers each as a `bid_<card>` option with a fixed description, so the question can never offer a card the player does not hold. A turn with only one card left in hand is resolved directly in Python instead (`docs/backends.md`, "Single-option Choice"): a `Choice` needs at least two options to be worth asking, and `helpers.play_game` never builds one for the forced last card of a game.


In [ ]:
legal = helpers.legal_actions_from_hand(first_turn.fields["hand"])
questions = helpers.build_questions(legal)
bid = questions["bid"]
print(bid.instructions)
for name, description in bid.criteria.items():
    print(f"  {name:8s} {description}")

## One answer up close

Before any aggregate, look at one typed answer. A `Choice` answer holds the chosen option, a `probabilities` entry for every option Python offered, and a `confidence` derived from that distribution: the top probability rescaled by the number of options on offer, `(p_max - 1/n) / (1 - 1/n)`. Two turns can report the same top probability and different confidence whenever they do not offer the same number of options -- the point the "Confidence by number of options" table later in this notebook returns to.


In [ ]:
result = backend.decide(state, questions)
answer = result["bid"]
show_answer(answer)

In [ ]:
plot_answer_probabilities(answer, title="One Choice answer")

## Python's part

Jev supplies the per-turn judgment; two rules in `helpers.py` hold whatever it answers. First, `CardDuel.step` (a `jev_cookbook.simulation.Simulator`) rejects any action that is not in the hand it just enumerated, so a model answer can never become an illegal move -- the "Evaluation" section demonstrates this held on every turn this notebook actually played, not only that the type system allows it. Second, `gate_turn` is this recipe's one review rule: a turn is auto-played with the model's own choice only when its confidence clears a gate chosen here, on `validation`; a less confident turn is still played -- a card game cannot simply skip a turn -- but with the rule-based action instead, and is recorded for [review](../../docs/glossary.md#review) with the reason `jev_cookbook` reserves for a confidence gate, `"confidence below the threshold"`. This is the recipe's only review branch, so it is a confidence-only rule: `jev_cookbook.evaluation.select_confidence_threshold` and `evaluate_selective` describe it directly, with no need for the outcomes-based functions a richer rule would need.

`select_confidence_threshold(..., min_coverage=0.75)` picks the lowest-risk gate among those that still auto-play at least three-quarters of validation turns, which is what "min_coverage" means here: a cookbook convention in favour of a coverage floor over forcing perfect validation accuracy, which a handful of mislabelled turns in a small validation set could make infeasible.


In [ ]:
MIN_COVERAGE = 0.75  # the confidence gate's objective, re-derived independently in tests/

val_examples = select_split(examples, "validation")
val_answers = [
    helpers.decide_turn(backend, e.fields, helpers.legal_actions_from_hand(e.fields["hand"]))
    for e in val_examples
]
val_gold = [labels[e.id] for e in val_examples]
val_correct = [a.choice == g for a, g in zip(val_answers, val_gold, strict=True)]
val_confidence = [a.confidence for a in val_answers]
threshold = select_confidence_threshold(val_correct, val_confidence, min_coverage=MIN_COVERAGE)
print(f"validation turns: {len(val_examples)}{selection}{check}")
print(
    f"accuracy of the raw choice against the rule-based gold action: "
    f"{accuracy(val_gold, [a.choice for a in val_answers]):.4f}{selection}{check}"
)
print(
    f"confidence gate chosen for min_coverage={MIN_COVERAGE}, frozen from here on: "
    f"{threshold:.4f}{selection}{check}"
)

for turn, raw_answer in ((first_turn, answer), (second_turn, None)):
    # The first turn's answer is the one already shown above, so it is not asked a second time.
    raw_answer = raw_answer or helpers.decide_turn(
        backend, turn.fields, helpers.legal_actions_from_hand(turn.fields["hand"])
    )
    fallback = helpers.rule_based_action(turn.fields["prize"], turn.fields["hand"])
    decision = helpers.gate_turn(raw_answer, fallback, threshold)
    print(
        f"{turn.id}: {raw_answer.choice} at confidence {raw_answer.confidence:.4f} "
        f"-> {decision.outcome} ({decision.reason or 'auto-played'}), played {decision.action}"
    )

## Evaluation

Two different things are evaluated, over two different quantities, and this section keeps them apart (one game is a task-level *unit*; one decided turn within it is a scored *example*). First, per question: the confidence gate was chosen above, on `validation`; this section applies it to `test` and reports the per-turn [accuracy](../../docs/glossary.md#accuracy), [coverage](../../docs/glossary.md#coverage) and [risk](../../docs/glossary.md#risk) of the gated turns, comparing the model's raw choice against the rule-based action recorded as each turn's [gold label](../../docs/glossary.md#gold-label). Second, at task level: three policies -- a random legal policy, the rule-based policy, and the confidence-gated model policy -- each play every seeded game end to end, and the section reports who wins, over both baselines this recipe names (the random legal policy and the rule-based policy), with an independent proof that none of the three ever played an illegal action. In an offline run every number here, `validation` and `test` alike, is a check that the pipeline works, and says so; every `validation` number also carries the selection-step label, because `test` is used once, to report, and `validation` never is.


In [ ]:
print(
    f"validation, {len(val_examples)} turns across {len(set(e.fields['game_seed'] for e in val_examples))} games"
    f"{selection}{check}"
)
print(
    f"accuracy of the raw choice: {accuracy(val_gold, [a.choice for a in val_answers]):.4f}{selection}{check}"
)

In [ ]:
test_examples = select_split(examples, "test")
test_answers = [
    helpers.decide_turn(backend, e.fields, helpers.legal_actions_from_hand(e.fields["hand"]))
    for e in test_examples
]
test_gold = [labels[e.id] for e in test_examples]
test_correct = [a.choice == g for a, g in zip(test_answers, test_gold, strict=True)]
test_confidence = [a.confidence for a in test_answers]
test_games = sorted({e.fields["game_seed"] for e in test_examples})

print(
    f"test, {len(test_examples)} turns across {len(test_games)} games, gate {threshold:.4f} frozen{check}"
)
print(
    f"accuracy of the raw choice: {accuracy(test_gold, [a.choice for a in test_answers]):.4f}{check}"
)
gated = evaluate_selective(test_correct, test_confidence, threshold)
print(
    f"auto-played (confidence >= gate): {gated.n_answered} of {gated.n_total} turns, "
    f"coverage {gated.coverage:.4f}{check}"
)
print(f"accuracy of the auto-played turns: {gated.accuracy:.4f}{check}")
print(f"risk of the auto-played turns: {gated.risk:.4f}{check}")
print(
    f"sent to review for {helpers.REASON_LOW_CONFIDENCE!r}: {gated.n_total - gated.n_answered} of {gated.n_total}{check}"
)

In [ ]:
all_actions = [f"bid_{card}" for card in range(1, helpers.N_CARDS + 1)]
matrix = confusion_matrix(test_gold, [a.choice for a in test_answers], labels=all_actions)
width = max(len(label) for label in matrix.labels)
leader_width = max(width + len("gold "), len("predicted ->"))
header = "  ".join(f"{label:>{width}s}" for label in matrix.labels)
print(f"Test confusion matrix, gold rows by predicted columns{check}:")
print(f"  {'predicted ->':<{leader_width}s}  {header}{check}")
for gold_label, row in zip(matrix.labels, matrix.matrix.tolist(), strict=True):
    counts = "  ".join(f"{count:{width}d}" for count in row)
    leader = f"gold {gold_label:<{width}s}"
    print(f"  {leader:<{leader_width}s}  {counts}{check}")
plot_confusion_matrix(matrix, title=f"Test split, {len(test_examples)} turns{check}")

### Confidence by number of options

A turn's `confidence` is rescaled by how many options it offered (`(p_max - 1/n) / (1 - 1/n)`), and the number of options this recipe offers shrinks every turn as the hand empties, from 5 down to 2 among the decided turns. A confidence of 0.60 reported on a 2-option turn and a confidence of 0.60 reported on a 5-option turn are not the same scale, so the pooled mean below is disclosed rather than read as one number, and the table beside it reports each option-set size on its own.


In [ ]:
by_arity = {}
for example, test_answer in zip(test_examples, test_answers, strict=True):
    n_options = len(example.fields["hand"])
    by_arity.setdefault(n_options, []).append(test_answer.confidence)

pooled_mean = sum(test_confidence) / len(test_confidence)
print(
    f"pooled mean confidence across every decided test turn, options not held constant: "
    f"{pooled_mean:.4f}{check} (not one scale -- see the per-option-count table below)"
)
for n_options in sorted(by_arity):
    values = by_arity[n_options]
    mean_confidence = sum(values) / len(values)
    print(
        f"  {n_options} options on offer: n={len(values)}, mean confidence {mean_confidence:.4f}{check}"
    )

In [ ]:
curve = selective_curve(test_correct, test_confidence)
for t, c, a, r in zip(curve.thresholds, curve.coverage, curve.accuracy, curve.risk, strict=True):
    print(f"gate >= {t:.4f}: coverage {c:.4f}, accuracy {a:.4f}, risk {r:.4f}{check}")
plot_risk_coverage(curve, label="test", title=f"Test split risk-coverage{check}")

In [ ]:
print("Test turns the raw choice got wrong, and what the confidence gate did with them:")
for example, test_answer, gold in zip(test_examples, test_answers, test_gold, strict=True):
    if test_answer.choice != gold:
        fallback = helpers.rule_based_action(example.fields["prize"], example.fields["hand"])
        decision = helpers.gate_turn(test_answer, fallback, threshold)
        print(
            f"  {example.id}: gold {gold}, answered {test_answer.choice} "
            f"(confidence {test_answer.confidence:.4f}) -> {decision.outcome}, played {decision.action}"
        )

The cell above is why the confidence gate is not decorative: several wrong turns clear it anyway (the non-zero risk printed earlier), and several right turns miss it (sent to review even though the raw choice was correct). A low-confidence turn is never left unplayed -- the game still needs a card bid every turn -- so Python substitutes the rule-based action rather than the model's own uncertain pick, and records the substitution for review rather than silently acting on an unconfident answer.


### Task level: playing whole seeded games

Each turn above was scored on its own, against the rule-based action recorded for it. This section instead plays each seeded game end to end under one policy at a time -- the random legal policy, the rule-based policy, and the confidence-gated model policy built from `gate_turn` and the frozen gate above -- and reports which policy wins, over both baselines this recipe names (the random legal policy and the rule-based policy). A game is a task-level *unit*; the turns inside it are the *examples* scored above, so this section's counts are over games, never turns.


In [ ]:
VALIDATION_SEEDS = tuple(sorted({e.fields["game_seed"] for e in val_examples}))
TEST_SEEDS = tuple(test_games)

gated_policy = helpers.make_gated_policy(backend, threshold)
raw_policy = helpers.make_raw_policy(backend)  # shown beside the gate, not a baseline


def random_policy_for(seed):
    return helpers.make_random_policy(f"{helpers.RECIPE_SLUG}:random:{seed}")


POLICIES = {
    "random legal policy": lambda seed: random_policy_for(seed),
    "rule-based policy": lambda seed: helpers.rule_based_policy,
    "confidence-gated model policy": lambda seed: gated_policy,
    "model policy, ungated (not a baseline)": lambda seed: raw_policy,
}


def play_all(seeds, policy_factory):
    return [helpers.play_game(policy_factory(seed), seed=seed) for seed in seeds]


def summarize(results):
    n = len(results)
    wins = sum(r.winner == "player" for r in results)
    losses = sum(r.winner == "opponent" for r in results)
    ties = sum(r.winner == "tie" for r in results)
    mean_score = sum(r.player_score for r in results) / n
    return wins, losses, ties, wins / n, mean_score


validation_results = {
    name: play_all(VALIDATION_SEEDS, factory) for name, factory in POLICIES.items()
}
test_results = {name: play_all(TEST_SEEDS, factory) for name, factory in POLICIES.items()}

print(f"validation games: {len(VALIDATION_SEEDS)}{selection}{check}")
for name, results in validation_results.items():
    wins, losses, ties, win_rate, mean_score = summarize(results)
    note = (
        " (constant for every seed tried against this fixed opponent, a structural property "
        "of the two fixed strategies, not a result)"
        if name == "rule-based policy"
        else ""
    )
    print(
        f"  {name}: {wins}-{losses}-{ties}, win rate {win_rate:.4f}, mean score {mean_score:.4f}"
        f"{selection}{check}{note}"
    )

print(f"test games: {len(TEST_SEEDS)}, gate {threshold:.4f} frozen{check}")
for name, results in test_results.items():
    wins, losses, ties, win_rate, mean_score = summarize(results)
    note = (
        " (constant for every seed tried against this fixed opponent, a structural property "
        "of the two fixed strategies, not a result)"
        if name == "rule-based policy"
        else ""
    )
    print(
        f"  {name}: {wins}-{losses}-{ties}, win rate {win_rate:.4f}, mean score {mean_score:.4f}{check}{note}"
    )

### Zero illegal actions, independently reproved

`helpers.play_game` already checks every policy's chosen action against the legal actions Python enumerated for that turn before playing it, and `CardDuel.step` checks again underneath -- a test in `tests/` feeds a policy that never names a legal action and confirms the simulator rejects it. The cell below is the stronger, independent half: it does not trust that nothing raised while this notebook ran it. It takes every trajectory the games above actually produced, resets a fresh `CardDuel` from the same seed, and replays each logged action through `Simulator.replay`, which re-checks legality (and that every observation and outcome still matches) from scratch. A single illegal action anywhere would raise before the count below is reached.


In [ ]:
total_replayed = 0
for _name, results in test_results.items():
    for result in results:
        fresh = helpers.CardDuel(seed=result.seed, n_cards=helpers.N_CARDS)
        fresh.replay(result.trajectory, seed=result.seed)  # raises on any illegal action
        total_replayed += len(result.trajectory)
expected = len(TEST_SEEDS) * len(POLICIES) * helpers.N_CARDS
assert total_replayed == expected
print(
    f"{total_replayed} of {expected} expected test-game actions independently replayed "
    f"through Simulator.replay, none rejected as illegal{check}"
)

## What was and was not measured

This run computes its answers with a seeded script written for this recipe (`helpers.script`), never a call to Jev, which the cell below states from `backend.mode`. Nothing here says anything about Jev's quality, speed or cost. `N` below counts the scored turns in each split; the two `demo` turns shown earlier carry no gold label and are not counted.


In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(val_examples)} validation and {len(test_examples)} test turns{check}")
else:
    dates = ", ".join(getattr(backend, "recorded_dates", ()) or ()) or "this run"
    print(f"Provenance: {backend.mode}, model {backend.model}")
    print(f"Captured: {dates}")
    print(f"N: {len(val_examples)} validation and {len(test_examples)} test turns")

## Next steps

- Change `helpers.P_MISTAKE` in `helpers.py` and run `pytest recipes/36-card-game-action-selection` or this notebook: the scripted model answers differently, and the confidence gate chosen on `validation` moves with it.
- Raise `helpers.N_CARDS` and rerun `build_fixtures.py`: longer games mean more decided turns per game, and a wider spread of option-set sizes for the "Confidence by number of options" table to show.
- Give the opponent in `helpers.CardDuel` a different fixed rule and see whether the rule-based baseline still wins every seeded game, or whether its win rate stops being a structural certainty.
- Neighbouring recipes: [`08-faq-selection`](../08-faq-selection/) and [`27-function-selection`](../27-function-selection/), which also pick one option from a fixed, Python-built list with a `Choice` question.

A neighbour link above is a folder link, `../NN-slug/`: link only a recipe whose `notebook.ipynb` is already committed on `main`, since a folder link to one that is not yet published would 404 on GitHub (`docs/recipe-template.md`).
